# Gold: publish reporting tables

Publish a cleaned dataset and an ingestion summary in the Gold schema. In Power BI, connect to a Databricks SQL Warehouse and select `superpower.gold.pbi_dataset` or `superpower.gold.ingestion_summary`. This is a technical starter model; business-specific dimensions, measures, and data types should be added after the source schema and report questions are defined.

In [ ]:
from pyspark.sql import functions as F

CATALOG = "superpower"
SILVER_TABLE = f"{CATALOG}.silver.csv_clean"
PBI_TABLE = f"{CATALOG}.gold.pbi_dataset"
INGESTION_SUMMARY_TABLE = f"{CATALOG}.gold.ingestion_summary"

silver_df = spark.table(SILVER_TABLE)
pbi_df = (
    silver_df
    .withColumnRenamed("_source_file", "source_file")
    .withColumnRenamed("_ingested_at", "ingested_at")
    .withColumn("ingestion_date", F.to_date("ingested_at"))
    .withColumn(
        "source_file_name",
        F.element_at(F.split(F.col("source_file"), \
), -1),
    )
)

ingestion_summary_df = (
    pbi_df.groupBy("ingestion_date", "source_file", "source_file_name")
    .agg(F.count(F.lit(1)).alias("row_count"))
)

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.gold")
for dataframe, table_name in [
    (pbi_df, PBI_TABLE),
    (ingestion_summary_df, INGESTION_SUMMARY_TABLE),
]:
    (
        dataframe.write.format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(table_name)
    )

print(f"Power BI dataset: {PBI_TABLE}")
print(f"Ingestion summary: {INGESTION_SUMMARY_TABLE}")
display(spark.table(PBI_TABLE).limit(20))
display(spark.table(INGESTION_SUMMARY_TABLE))